# Inspect SCA Target

QA/QC of the snow-covered area calibration target written by
`targets/sca.py` to `<project>/targets/sca_targets.nc` (and the
optional `sca_targets_nn_filled.nc` companion).

The SCA target is a per-HRU per-**day** `(lower_bound, upper_bound)`
range as a **fraction in [0, 1]** (PRMS `snowcov_area` units), built
single-source from MOD10C1 v061 via the CI-bounded formula from
`PRMSobjfun.f90:calcSCA` (`docs/references/PRMSobjfun.f90` lines
1052-1061):

    sca_obs = Day_CMG_Snow_Cover / 100        # fractional 0-1
    ci      = Day_CMG_Clear_Index / 100        # fractional 0-1
    where ci_hru >= ci_threshold (default 0.70):
        lower = ci * sca_obs
        upper = lower + (1 - ci)               # algebraically capped at 1
    where ci_hru < ci_threshold:
        lower = upper = NaN
    July / August: lower = upper = 0 (calcSCA forced-zero rule).

Because the bound is built from a single source, the `n_sources`
flag is binary `{0, 1}` rather than the 0-4 range of multi-source
targets like SWE. The bound is NaN exactly when `n_sources == 0`.

This notebook checks:

- File schema and global metadata (period, `ci_threshold`,
  `summer_zero_months`, fabric SHA, source string).
- Per-time `n_sources` coverage (monthly mean) and at-time NaN map.
- `lower_bound`, `upper_bound`, and range (= `1 - ci_hru`) at peak
  winter day (`TARGET_DATE`, default 2010-02-15).
- Recovered HRU-mean CI distribution at TARGET_DATE — sanity-checks
  the gate behaviour.
- July / August forced-zero floor — confirms `upper.max() == 0` for
  every Jul/Aug day in the build, by year.
- CONUS area-weighted lower / upper monthly series — the seasonal
  cycle should sit at zero Jul-Aug and rise toward 1.0 in winter.
- Representative-HRU daily series across OR snow-bearing regions
  (Cascades / Coast Range / Steens / Willamette).
- NN-fill: which HRU-times were filled and whether the filled
  distribution tracks the unfilled one.

Companion to `inspect_aggregated_mod10c1.ipynb` (per-pixel CI-gated
SCA fraction before HRU aggregation) and the SCA section of the
per-fabric slide decks under `docs/presentations/`.

## Conventions

- HRU dim name follows `fabric.id_col` from the project config
  (e.g. `nat_hru_id` for the GFv2 fabric, `nhm_id` for OR).
- Bounds are **fractional [0, 1]** — CF `units: '1'`. Multiply by
  100 only for human-readable percent reporting (not stored on disk).
- `TARGET_DATE` (set below) drives the at-time choropleth panels.
  Default is **mid-February**, near peak CONUS snow extent — the
  cleanest spread across snow-bearing HRUs because seasonal snowpack
  is broadest and cloud cover is moderate. Switch to a summer date
  to inspect the forced-zero regime instead.
- The SCA target NC is ~180 MB (daily × 16k HRUs × 3 variables as
  float32 + int8), so no `TIME_WINDOW` slicing is needed — the full
  range loads comfortably under the default render kernel.
- Recovered HRU-mean CI: by construction `upper - lower = 1 - ci`
  (algebraic identity from `calcSCA`), so any finite bound exposes
  the HRU's CI on that day as `1 - (upper - lower)`. Useful for the
  CI-distribution diagnostic cell below.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    area_weighted_series,
    discover_target_nc,
    load_fabric,
    load_project_paths,
    load_representative_points,
    lookup_hrus_by_points,
    n_sources_per_time,
    nan_hru_count,
    open_target_nc,
    plot_categorical_choropleth,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/targets/<project>/*.png
import _helpers
_helpers.SAVE_FIGURES = True
_helpers.PROJECT = PROJECT_DIR.name

TARGET = "sca"
TARGET_DATE = "2010-02-15"  # near-peak CONUS snow extent

# 4-yr legibility window for the representative-HRU time series plot.
# The full 25-yr daily series is too dense to read individual upper /
# lower bound values; 2008-2011 spans 4 full water years mid-period
# with complete data quality and includes the OR-significant 2010-11
# deep snowpack year. Only the representative_series cell uses this
# window; every other diagnostic uses the full file.
REPSERIES_WINDOW = ("2008-01-01", "2011-12-31")

# Fall back to a CONUS snow-belt grid if the project has no
# representative_points block. OR's config.yml provides four PNW
# snow-bearing headwater points (Willamette / Deschutes / Umpqua /
# Rogue) under the `sca:` key — those are picked up automatically
# when --project-dir points at or-spatial-targets.
REPRESENTATIVE_POINTS = load_representative_points(PROJECT_DIR, TARGET) or {
    "Sierra Nevada (deep maritime snowpack)": (-119.0, 38.0),
    "Colorado Rockies (alpine continental)": (-106.8, 39.5),
    "Cascades (PNW maritime)": (-121.5, 47.0),
    "Adirondacks (eastern shallow)": (-74.0, 44.0),
}

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)
id_dim = fabric_cfg["id_col"]

print(f"Project:   {project_dir}")
print(f"Datastore: {datastore_dir}")
print(f"Fabric:    {fabric_cfg['path']} ({len(fabric)} HRUs, id_col={id_dim!r})")
print(f"Target date: {TARGET_DATE}")
print(f"Rep-series window: {REPSERIES_WINDOW[0]} → {REPSERIES_WINDOW[1]}")

## Open and summarise the target NCs

`discover_target_nc` finds both the unfilled (`sca_targets.nc`) and
NN-filled (`sca_targets_nn_filled.nc`) variants if they exist.
Either may be `None` — this cell prints a clear skip message and
exits if the unfilled file is missing (the target has not been built
for this project yet).

In [ ]:
raw_path, filled_path = discover_target_nc(project_dir, TARGET)

if raw_path is None:
    print(f"SKIP: {TARGET}_targets.nc not found at {project_dir / 'targets'}.")
    print(f"      Run `pixi run run-sca -- --project-dir {project_dir}` first.")
    raise SystemExit

ds_raw = open_target_nc(raw_path)
print(f"Loaded raw target: {raw_path.name}  ({raw_path.stat().st_size / 1e6:.1f} MB on disk)")
print(f"  timesteps: {ds_raw.sizes.get('time', 0)}")

ds_filled = None
if filled_path is not None:
    ds_filled = open_target_nc(filled_path)
    print(f"Loaded NN-filled:  {filled_path.name}  ({ds_filled.sizes.get('time', 0)} timesteps)")
else:
    print("NN-filled variant absent (set `nn_fill: true` in config to produce it).")

## Schema and global metadata

Global attrs carry the provenance you'd want at calibration time:
the active `period` (the build's actual coverage, not the
`catalog/variables.yml` default), `ci_threshold` (the HRU-mean CI
gate that produced the NaN mask), `summer_zero_months` (the
calcSCA-mandated Jul/Aug forced-zero list), the fabric SHA-256, and
the comma-separated `source` string (always `mod10c1_v061` for SCA).

Per-variable units are `'1'` (CF dimensionless for the fractional
bounds; `'1'` also for the `n_sources` count, with `flag_values =
[0, 1]` and `flag_meanings = "none one"`).

In [ ]:
print(ds_raw)
print()
print("=== Global attrs ===")
for k, v in ds_raw.attrs.items():
    print(f"  {k:<22} {v}")

print()
print("=== Per-variable units / long_name ===")
for v in ("lower_bound", "upper_bound", "n_sources"):
    units = ds_raw[v].attrs.get("units", "(no units attr)")
    long_name = ds_raw[v].attrs.get("long_name", "")
    print(f"  {v:<14} units={units!r}  long_name={long_name!r}")

## Per-time coverage

`n_sources_per_time` returns the per-day count of HRUs at each flag
value — `{0, 1}` for SCA since there's a single source. Daily
resolution over a 25-year run is noisy, so we **resample to monthly
mean** before plotting.

Two diagnostics:

- The `n=0` series is the count of all-NaN HRUs at that timestep —
  the cells the NN-fill targets. For SCA this is structurally large
  in winter (cloud cover failing the CI gate) and structurally near-
  total in Jul/Aug (forced-zero rule applies to passing HRUs, but the
  CI gate still drops cloud-blocked HRUs to NaN).
- The `n=1` series is the count of CI-passing HRUs. A strong
  seasonal cycle is expected — clear skies are most common in the
  warm dry season.

In [ ]:
cov = n_sources_per_time(ds_raw)
cov_monthly = cov.resample("MS").mean()
print(cov_monthly.describe().T[["mean", "std", "min", "max"]])

fig, ax = plt.subplots(figsize=(11, 4))
cov_monthly.plot(ax=ax)
ax.set_xlabel("Time (monthly mean of daily counts)")
ax.set_ylabel("HRU count")
ax.set_title(f"Per-day n_sources distribution \u2014 {TARGET} target (monthly mean)")
ax.legend(title="flag value", loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_coverage_timeseries")
plt.show()

## n_sources map at TARGET_DATE

Categorical choropleth, colours match the `flag_values` attr on the
`n_sources` variable. NaN HRUs (where the time slice has no
CI-passing source) plot in light grey; the legend reports the count
of HRUs at each flag value so coverage anomalies are visible at a
glance.

On a peak-snow winter day, expect snow-bearing HRUs to show `1`
(CI-passing) and cloudy-day HRUs to show `0` (CI-failing). The map
is essentially a cloud-cover proxy for that day.

In [ ]:
ns = ds_raw["n_sources"].sel(time=TARGET_DATE).to_pandas()

categories = {
    0: ("0 (CI gate failed)", "crimson"),
    1: ("1 (CI passed)", "midnightblue"),
}

fig, ax = plt.subplots(figsize=(11, 7))
plot_categorical_choropleth(
    ax,
    fabric,
    ns,
    categories=categories,
    title=f"n_sources at {TARGET_DATE} \u2014 {TARGET} target",
)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_n_sources_map")
plt.show()

## Lower / upper / range maps at TARGET_DATE

Three side-by-side panels:

- `lower_bound` = `ci_hru * sca_obs` — the confidence-weighted
  "definitely snow" floor (fraction 0-1).
- `upper_bound` = `lower + (1 - ci_hru)` — the bound's ceiling,
  algebraically capped at 1.
- `range = upper - lower = 1 - ci_hru` — width of the bound,
  inversely proportional to CI confidence. High-CI HRUs collapse to
  a near-point bound; low-CI HRUs (just above the 0.70 gate) span
  the full [0, 0.3] window. This is the per-HRU per-day spread the
  PRMS calibrator is asked to land inside.

Colour scale: bounds anchored on `[0, 1]` (the full fractional
domain); range anchored on `[0, 0.3]` (the maximum width any
CI-passing HRU can achieve, at `ci = 0.70`).

In [ ]:
lb = ds_raw["lower_bound"].sel(time=TARGET_DATE).to_pandas()
ub = ds_raw["upper_bound"].sel(time=TARGET_DATE).to_pandas()
rng = ub - lb

units = ds_raw["lower_bound"].attrs.get("units", "1")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
plot_hru_choropleth(
    axes[0], fabric, lb, vmin=0, vmax=1, cmap="Blues",
    title=f"lower_bound\n{TARGET_DATE} | fraction", units=units,
)
plot_hru_choropleth(
    axes[1], fabric, ub, vmin=0, vmax=1, cmap="Blues",
    title=f"upper_bound\n{TARGET_DATE} | fraction", units=units,
)
plot_hru_choropleth(
    axes[2], fabric, rng, vmin=0, vmax=0.3, cmap="OrRd",
    title=f"range = upper - lower = (1 - ci_hru)\n{TARGET_DATE} | fraction",
    units=units,
)
fig.suptitle(f"{TARGET} target bounds \u2014 {TARGET_DATE}", fontsize=13, y=1.02)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_bounds_map")
plt.show()

print(f"lower CONUS area-weighted mean: {area_weighted_mean(lb, fabric):.4f} (fraction)")
print(f"upper CONUS area-weighted mean: {area_weighted_mean(ub, fabric):.4f} (fraction)")
print(f"range CONUS area-weighted mean: {area_weighted_mean(rng, fabric):.4f} (fraction)")

## NaN HRU coverage at TARGET_DATE

HRUs where `n_sources == 0` (CI gate failed or, in Jul/Aug, the
forced-zero rule applies *plus* the gate failed). For SCA this is
structurally larger than for multi-source targets like SWE because
any single cloudy day blanks an HRU — no other source backstops it.

Expect ~40-70% coverage on a typical winter day, depending on cloud
cover. These are the HRUs that nearest-neighbour fill targets.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
plot_nan_hrus(
    ax, fabric, lb,
    title=(
        f"NaN HRUs (red) \u2014 {TARGET_DATE} | "
        f"{nan_hru_count(lb)} of {len(fabric)} "
        f"({100 * nan_hru_count(lb) / len(fabric):.2f}%)"
    ),
)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_nan_map")
plt.show()

## Recovered HRU-mean CI distribution at TARGET_DATE

By construction `upper - lower = 1 - ci_hru` for any finite bound,
so we can recover the HRU-mean CI distribution from the target NC
without re-reading the aggregated mod10c1_v061 source.

Diagnostic value:

- Distribution mass below `ci_threshold` (vertical line) is the
  population that was gated out — should be **empty** by
  construction (gated HRUs are NaN, not in this histogram).
- Concentration near `ci = 1` (rightmost bar) is the high-confidence
  population — clear-sky days dominate.
- Spread between `[ci_threshold, 1]` shows how much CI variability
  the surviving HRUs carry. Tight near 1 = mostly fully-clear; spread
  toward 0.70 = many marginal-CI HRUs survived (wider bounds, more
  calibration slack).

In [ ]:
ci_threshold = ds_raw.attrs.get("ci_threshold", 0.70)
ci_recovered = (1.0 - rng).dropna()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(ci_recovered.values, bins=np.linspace(0.5, 1.0, 51), color="steelblue", edgecolor="white")
ax.axvline(ci_threshold, color="crimson", linestyle="--", label=f"ci_threshold = {ci_threshold}")
ax.set_xlabel("Recovered HRU-mean CI (= 1 - (upper - lower))")
ax.set_ylabel("HRU count")
ax.set_title(
    f"Recovered HRU-mean CI distribution \u2014 {TARGET_DATE} "
    f"(N={len(ci_recovered)} finite of {len(fabric)} HRUs)"
)
ax.legend()
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_ci_distribution")
plt.show()

print(f"Recovered CI quantiles (0.10 / 0.25 / 0.50 / 0.75 / 0.90):")
print(ci_recovered.quantile([0.10, 0.25, 0.50, 0.75, 0.90]).round(4).to_string())

## July / August forced-zero floor

`PRMSobjfun.f90:calcSCA` forces `(lower, upper) = (0, 0)` for every
CI-passing HRU during July and August (the months when the original
TM 6-B10 fabrics have no expected snowpack). This is a deliberate
calibration choice — it tells the optimiser that any modelled
summer snow is observation-incompatible.

Check: `upper.max()` across all (HRU, time) in Jul/Aug should be
exactly `0.0` for every year. If it isn't, the forced-zero rule
didn't fire and the target is non-faithful to calcSCA.

**OR-specific caveat.** On the Cascades crest, glaciers and
permanent snowfields *do* hold late-summer snow. The forced-zero
floor is therefore observationally questionable for OR's highest-
elevation HRUs — see the related discussion in the OR slide deck.
This notebook reports the floor as-built; whether to keep, override,
or per-HRU disable it is a calibration-team decision.

In [ ]:
ub_all = ds_raw["upper_bound"]
ja_mask = ub_all.time.dt.month.isin([7, 8])
ub_ja = ub_all.where(ja_mask, drop=True)

per_year_max = (
    ub_ja
    .groupby("time.year")
    .max()
    .max(dim=id_dim)
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(per_year_max.index, per_year_max.values, color="steelblue")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("Calendar year")
ax.set_ylabel("max(upper_bound) across Jul+Aug, all HRUs")
ax.set_title(
    f"Jul / Aug forced-zero floor check \u2014 should be exactly 0.0 every year"
)
ax.set_ylim(-0.05, max(0.05, float(per_year_max.max()) * 1.1))
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_jul_aug_floor")
plt.show()

summer_zero_months = ds_raw.attrs.get("summer_zero_months", "7,8")
print(f"summer_zero_months attr: {summer_zero_months!r}")
print(f"Jul/Aug upper_bound max (all years, all HRUs): {float(ub_ja.max()):.6f}")
print(f"Per-year Jul/Aug max (expect all 0):")
print(per_year_max.to_string())

## CONUS area-weighted lower / upper time series (monthly mean)

`area_weighted_series` reduces the (time, hru) bound arrays to a
per-timestep CONUS-mean weighted by EPSG:5070 polygon area. Resampled
to **monthly mean** for legibility — the underlying NC is daily.

Lower / upper plotted together form an envelope; envelope width =
`1 - ci_hru` (area-weighted), so a narrowing envelope means clearer
skies on average.

Expect:

- A strong seasonal cycle (peak ~Jan-Mar, near-zero ~Aug) — SCA is
  the most strongly seasonal of all six calibration targets.
- The envelope sits exactly on `0` for July and August (forced-zero
  rule applied to every passing HRU).

In [ ]:
lb_series = area_weighted_series(ds_raw["lower_bound"], fabric, id_dim)
ub_series = area_weighted_series(ds_raw["upper_bound"], fabric, id_dim)
lb_m = lb_series.resample("MS").mean()
ub_m = ub_series.resample("MS").mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(
    lb_m.index, lb_m.values, ub_m.values,
    color="steelblue", alpha=0.25, label="lower\u2013upper envelope",
)
ax.plot(lb_m.index, lb_m.values, color="steelblue", lw=1, label="lower (monthly mean)")
ax.plot(ub_m.index, ub_m.values, color="darkblue", lw=1, label="upper (monthly mean)")
ax.set_ylim(0, 1)
ax.set_xlabel("Time")
ax.set_ylabel("Area-weighted CONUS mean (fraction)")
ax.set_title(f"{TARGET} target \u2014 CONUS area-weighted bounds (daily \u2192 monthly mean)")
ax.legend(loc="upper right")
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_conus_series")
plt.show()

## Representative HRU time series (daily)

OR-config supplies four PNW snow-bearing regions (Cascades / Coast
Range / Steens / Willamette); other projects fall back to the
hardcoded CONUS-wide defaults.

`lookup_hrus_by_points` uses `gpd.sjoin` to resolve each (lon, lat)
to the containing HRU. Plotted at native **daily** cadence so
individual cloud-free snow days, melt-out timing, and the Jul/Aug
forced-zero floor stay visible. NaN runs (gaps in the lines) are
cloudy-day CI-gate failures.

In [ ]:
rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
print("Representative HRUs:", rep_hrus)

lb_at = (
    ds_raw["lower_bound"]
    .sel({id_dim: list(rep_hrus.values())})
    .sel(time=slice(*REPSERIES_WINDOW))
)
ub_at = (
    ds_raw["upper_bound"]
    .sel({id_dim: list(rep_hrus.values())})
    .sel(time=slice(*REPSERIES_WINDOW))
)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
    lb_h = lb_at.sel({id_dim: hru_id}).values
    ub_h = ub_at.sel({id_dim: hru_id}).values
    t = pd.DatetimeIndex(lb_at.time.values)
    ax.fill_between(t, lb_h, ub_h, color="steelblue", alpha=0.25)
    ax.plot(t, lb_h, color="steelblue", lw=0.5, label="lower")
    ax.plot(t, ub_h, color="darkblue", lw=0.5, label="upper")
    ax.set_title(f"{label} (HRU {hru_id})", fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_ylabel("SCA (fraction)")
    ax.legend(fontsize=8, loc="upper right")
fig.suptitle(
    f"{TARGET} target at representative HRUs (daily, {REPSERIES_WINDOW[0]} → {REPSERIES_WINDOW[1]})",
    fontsize=13, y=1.00,
)
plt.tight_layout()
save_figure(fig, f"{TARGET}_target_representative_series")
plt.show()

## NN-fill comparison

The companion `sca_targets_nn_filled.nc` reuses the same bounds but
fills NaN HRUs from up to `nn_max_candidates=10` nearest neighbours
(see `targets/_common.py:nn_fill_hru_nans_by_time`). The `nn_filled`
int8 flag marks each HRU-time as `0` (not filled) or `1` (filled).

Because the SCA NaN footprint is large (CI-gated cloud cover), the
NN-fill is doing more work here than for SWE — expect a substantial
fraction of HRUs to be filled on any given day. Some HRUs remain
NaN even after fill (no donor within 10 candidates) — typically
when the entire neighbourhood is cloudy together.

Two checks:

- **Filled-flag map at TARGET_DATE** — should track the NaN-HRU map
  closely (everywhere NaN that has a donor inside 10 candidates).
- **Distribution preservation (monthly mean)** — the area-weighted
  CONUS mean of the filled bounds should track the unfilled mean
  closely. Large gap = over-aggressive fill (donor HRUs too far).

In [ ]:
if ds_filled is None:
    print("SKIP: NN-filled variant not present.")
else:
    flag = ds_filled["nn_filled"].sel(time=TARGET_DATE).to_pandas()
    n_filled = int((flag == 1).sum())
    print(
        f"NN-filled at {TARGET_DATE}: {n_filled} HRUs filled "
        f"({100 * n_filled / len(fabric):.2f}%)"
    )

    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    plot_categorical_choropleth(
        axes[0], fabric, flag,
        categories={0: ("not filled", "lightgrey"), 1: ("filled", "crimson")},
        title=f"nn_filled flag at {TARGET_DATE}",
    )
    lb_f = ds_filled["lower_bound"].sel(time=TARGET_DATE).to_pandas()
    diff = (lb_f - lb).abs().fillna(lb_f)  # NaN in raw -> show filled value
    plot_hru_choropleth(
        axes[1], fabric, diff, vmin=0, vmax=0.5, cmap="OrRd",
        title=f"|lower_filled - lower_raw| at {TARGET_DATE}",
        units=units,
    )
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_nn_fill_map")
    plt.show()

    lb_f_series = area_weighted_series(ds_filled["lower_bound"], fabric, id_dim).resample("MS").mean()
    ub_f_series = area_weighted_series(ds_filled["upper_bound"], fabric, id_dim).resample("MS").mean()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(lb_m.index, lb_m.values, color="steelblue", lw=1, label="lower (raw, monthly mean)")
    ax.plot(lb_f_series.index, lb_f_series.values, color="steelblue", lw=1, ls="--", label="lower (NN-filled, monthly mean)")
    ax.plot(ub_m.index, ub_m.values, color="darkblue", lw=1, label="upper (raw, monthly mean)")
    ax.plot(ub_f_series.index, ub_f_series.values, color="darkblue", lw=1, ls="--", label="upper (NN-filled, monthly mean)")
    ax.set_ylim(0, 1)
    ax.set_xlabel("Time")
    ax.set_ylabel("Area-weighted CONUS mean (fraction)")
    ax.set_title(f"{TARGET} target \u2014 raw vs NN-filled CONUS series (monthly mean)")
    ax.legend()
    plt.tight_layout()
    save_figure(fig, f"{TARGET}_target_nn_fill_series")
    plt.show()

## Sanity check at TARGET_DATE

Bounds are fractional [0, 1] by construction (calcSCA returns
`lower = ci * sca_obs` and `upper = lower + (1 - ci)`, both in
[0, 1] for `ci, sca_obs \u2208 [0, 1]`). The only failure modes are:

- `upper < lower` anywhere (would indicate a sign error or NaN
  propagation bug — should be impossible).
- `lower < 0` or `upper > 1` (would indicate the units are still on
  the native 0-100 integer scale and the conversion was missed).
- Representative-HRU lower at peak winter near 0 in snow-bearing
  regions (would indicate per-pixel SCA observations are not landing
  inside the HRU's pre-aggregate mask).

Per `feedback_validate_magnitudes`: an order-of-magnitude miss
(e.g. peak winter Cascades lower < 0.05) is a smoking gun for a unit
or masking bug.

In [ ]:
print(f"Sanity check at {TARGET_DATE} \u2014 lower / upper SCA at representative HRUs")
print(f"{'Region':<55} {'lower':>10} {'upper':>10}")
print("-" * 78)
for label, hru_id in rep_hrus.items():
    lb_h = float(ds_raw["lower_bound"].sel({id_dim: hru_id}).sel(time=TARGET_DATE).values)
    ub_h = float(ds_raw["upper_bound"].sel({id_dim: hru_id}).sel(time=TARGET_DATE).values)
    print(f"{label:<55} {lb_h:>10.4f} {ub_h:>10.4f}")

print()
print(f"Global bounds check (entire NC):")
print(f"  lower min/max: {float(ds_raw['lower_bound'].min()):.6f} / {float(ds_raw['lower_bound'].max()):.6f}")
print(f"  upper min/max: {float(ds_raw['upper_bound'].min()):.6f} / {float(ds_raw['upper_bound'].max()):.6f}")
print(f"  (upper < lower) finite cells: {int(((ds_raw['upper_bound'] < ds_raw['lower_bound']) & ds_raw['upper_bound'].notnull()).sum())}")

In [ ]:
ds_raw.close()
if ds_filled is not None:
    ds_filled.close()